<a href="https://colab.research.google.com/github/RanaMagdyisaac/High-Energy-Particle-Classification-Model-/blob/main/High_Energy_Particle_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded = files.upload()

In [ ]:
file_name = list(uploaded.keys())[0]
raw_df = pd.read_csv(file_name, header=None)
print(" Raw Data: Head and Tail ")
display(pd.concat([raw_df.head(5), raw_df.tail(5)]))


In [ ]:
print(" Original Class Distribution ")
print(raw_df[10].value_counts())

# *Exploratory Data Analysis (EDA)*

Observations:Based on the raw data preview (Head and Tail), I have observed the following:Missing Headers: The dataset does not contain column names in the first row. Therefore, I need to manually assign the 11 feature names (fLength, fWidth, etc.) and the target class.Class Imbalance: The data is significantly imbalanced. There are 12,332 Gamma (g) events and only 6,688 Hadron (h) events.Required Action: To ensure an unbiased model, I must perform Data Balancing by downsampling the Gamma class to match the Hadron class size

In [ ]:
columns = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsym", "fM3Long", "fM3Trans", "fAlpha", "fDist", "class"]
df = pd.read_csv(file_name, names=columns)
df

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
print(df.duplicated().sum())

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print(df.duplicated().sum())

In [ ]:
df["class"] = (df["class"] == "g").astype(int)
df_g = df[df["class"] == 1]
df_h = df[df["class"] == 0]
df_g_balanced = df_g.sample(n=len(df_h), random_state=42)
df_balanced = pd.concat([df_g_balanced, df_h], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
print(df_balanced["class"].value_counts())

# Data Balancing Results:
After performing the downsampling and shuffling, the dataset is now perfectly balanced and ready for model training:Target Counts: Both classes (Gamma and Hadron) now have exactly 6573 samples each.Total Samples: The balanced dataset size is 13,146

Randomization: The data has been shuffled to prevent the models from learning the order of samples.Outcome: This prevents the classifier from being biased toward the majority class (Gamma), ensuring more reliable accuracy and recall metrics

In [ ]:
from sklearn.model_selection import train_test_split


X = df_balanced.drop("class", axis=1)
y = df_balanced["class"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

print(" Data Split:")
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

###Classification

In [ ]:
#Decision Tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


dt_model = DecisionTreeClassifier(random_state=42)
cv_scores_dt = cross_val_score(dt_model, X_train, y_train, cv=5)
dt_model.fit(X_train, y_train)


y_pred_dt = dt_model.predict(X_test)


accuracy = accuracy_score(y_test, y_pred_dt)
precision = precision_score(y_test, y_pred_dt)
recall = recall_score(y_test, y_pred_dt)
f1 = f1_score(y_test, y_pred_dt)
conf_matrix = confusion_matrix(y_test, y_pred_dt)

print(" Decision Tree Performance :")
print(f"Cross-Validation Mean Accuracy: {cv_scores_dt.mean():.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


plt.figure(figsize=(6,4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Decision Tree Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

 **Decision Tree Model Analysis:**

After training the Decision Tree classifier without parameter tuning, I obtained the following results on the test set:


Overall Accuracy: The model achieved an accuracy of 78.82%, correctly classifying the majority of gamma and hadron events.

Confusion Matrix Breakdown:

True Negatives (0,0): 1557 hadron events were correctly identified.

True Positives (1,1): 1534 gamma events were correctly identified.

Misclassifications: There were 413 False Positives and 440 False Negatives, showing that the model has a balanced error rate across both classes.


Balanced Performance: The Precision, Recall, and F1-Score are all very close to each other (around 0.77 - 0.78), which confirms that our Data Balancing step was successful

In [ ]:
#AdaBoost
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV


ada = AdaBoostClassifier(random_state=42)


param_grid_ada = {'n_estimators': [10, 50, 100,150]}


grid_ada = GridSearchCV(ada, param_grid_ada, cv=5, scoring='accuracy')
grid_ada.fit(X_train, y_train)


best_ada = grid_ada.best_estimator_
y_pred_ada = best_ada.predict(X_test)


print(f"Best n_estimators found: {grid_ada.best_params_['n_estimators']}")
print(f"Accuracy with best parameters: {accuracy_score(y_test, y_pred_ada):.4f}")

In [ ]:
from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt


precision_ada = precision_score(y_test, y_pred_ada)
recall_ada = recall_score(y_test, y_pred_ada)
f1_ada = f1_score(y_test, y_pred_ada)

print("--- AdaBoost (n_estimators=150) Full Report ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_ada):.4f}")
print(f"Precision: {precision_ada:.4f}")
print(f"Recall:    {recall_ada:.4f}")
print(f"F1-Score:  {f1_ada:.4f}")


print("\nDetailed Report:")
print(classification_report(y_test, y_pred_ada))


plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_ada), annot=True, fmt='d', cmap='YlGnBu')
plt.title('AdaBoost Confusion Matrix (Best Model)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

**AdaBoost Model Analysis:**

By tuning the n_estimators parameter to 150 using cross-validation, the AdaBoost model shows a significant improvement over the base Decision Tree:Performance Metrics: The model achieved an accuracy of 80.35%, with balanced Precision (80.85%) and Recall (79.58%).

Error Reduction: Compared to the previous model, AdaBoost reduced the number of misclassified events (False Positives and False Negatives), as seen in the Confusion Matrix.Consistency: The F1-Score of 0.80 indicates that the model is robust and handles both Gamma and Hadron classes effectively.

Impact of Tuning: Using 150 estimators allowed the model to build a stronger ensemble, confirming that parameter tuning was essential for this dataset

In [ ]:
#Random Forests
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {'n_estimators': [10, 50, 100, 150]}


grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='accuracy')
grid_rf.fit(X_train, y_train)


best_rf = grid_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test)


print(f" Random Forest Full Report ")
print(f"Best n_estimators: {grid_rf.best_params_['n_estimators']}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred_rf))


plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Purples')
plt.title('Random Forest Confusion Matrix (Best Model)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

**Random Forest Model Analysis :**

After conducting a grid search with cross-validation, the Random Forest classifier has emerged as the best-performing model so far among the tree-based methods:

Superior Performance: It achieved a notable accuracy of 85.80% using 150 estimators.

High Sensitivity: With a recall of 87.99%, it is currently the most effective model at capturing Gamma events.

Balanced F1-Score: The F1-score of 0.8612 indicates a strong balance, outperforming both AdaBoost and the single Decision Tree

In [ ]:
#Naïve Bayes
from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
cv_scoress_nb = cross_val_score(nb_model, X_train, y_train, cv=5)
nb_model.fit(X_train, y_train)


y_pred_nb = nb_model.predict(X_test)


print(" Naïve Bayes Report ")
print(f"Cross-Validation Mean Accuracy: {cv_scoress_nb.mean():.4f}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_nb):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_nb):.4f}")


print("\nDetailed Report:")
print(classification_report(y_test, y_pred_nb))

plt.figure(figsize=(6,4))
sns.heatmap(confusion_matrix(y_test, y_pred_nb), annot=True, fmt='d', cmap='Greens')
plt.title('Naïve Bayes Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

**Naïve Bayes Model Analysis:**

The Naïve Bayes model provided a very different perspective compared to the tree-based models:

High Sensitivity (Recall): It achieved a very high recall of 89.22%, meaning it is excellent at identifying almost all Gamma events.

Low Precision: However, it has a low precision of 60.50%, which means it has a high "False Positive" rate (it often misclassifies Hadrons as Gamma).

Overall Accuracy: With an accuracy of 65.42%, it ranks lower than the ensemble methods (Random Forest and AdaBoost).

Conclusion: While Naïve Bayes is computationally very fast and simple, it struggles with the complex feature relationships in this dataset, leading to more misclassifications

In [ ]:

comparison_data = {
    "Model": ["Decision Tree", "AdaBoost", "Random Forest", "Naïve Bayes"],
    "Accuracy": [0.7882, 0.8035, 0.8580, 0.6522],
    "Precision": [0.7764, 0.8001, 0.8265,  0.6050],
    "Recall": [0.7925, 0.7940, 0.8821,0.8906],
    "F1-Score": [0.7844, 0.7971, 0.8534,0.7205 ]
}


comparison_df = pd.DataFrame(comparison_data)


print("Final Model Comparison ")
display(comparison_df.sort_values(by="Accuracy", ascending=False))

In [ ]:

comparison_melted = comparison_df.melt(id_vars="Model", var_name="Metric", value_name="Score")


plt.figure(figsize=(12, 6))
sns.barplot(data=comparison_melted, x="Metric", y="Score", hue="Model", palette="viridis")


plt.title("Final Model Comparison: Accuracy, Precision, Recall, & F1-Score", fontsize=16)
plt.ylabel("Score (0.0 - 1.0)", fontsize=12)
plt.xlabel("Evaluation Metric", fontsize=12)
plt.ylim(0.5, 1.0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title="Classifier Model", loc='lower right')

plt.show()

**Final Comparative Analysis of All Models:**

This visualization provides a comprehensive summary of the model performance metrics:

The Absolute Winner: Random Forest (Emerald Green) is the top-performing model across almost all metrics, achieving the highest Accuracy (85.07%), Precision, and F1-Score. It demonstrates the best balance between precision and sensitivity.

The Sensitivity Paradox (Naïve Bayes): While Naïve Bayes (Light Green) shows the highest Recall, its Precision and Accuracy are the lowest. This indicates that while it captures most signals, it generates many "False Alarms" (False Positives).

The Power of Ensembles: Both Random Forest and AdaBoost (Ensemble methods) significantly outperform the individual Decision Tree, proving that combining multiple learners effectively reduces errors and improves stability.

Final Conclusion: Based on the overall performance and the F1-Score,**Random Forest** is the recommended model for this dataset due to its robustness and superior classification power.